In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
import re
import time as _time
from PIL.ImageColor import colormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from pathlib import Path
from ast import literal_eval


In [2]:
# --------------- Set up project root path  --------------- #
project_folder_name = "MFC2024" # Set this to the name of your project root folderS
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == project_folder_name), None)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from config.settings import *
from model.model import PEMFC_dyn
from model.coefficients import *
from config.initialize import *
from config.settings import *

In [ ]:
pola_tests_sim = {}
load_points    = [10, 20, 30, 45, 50]   # full sweep for reference plots
# Validation subset -- comment the next 3 lines out for the full grid
_VALIDATION_RHC = [0.5]
_VALIDATION_P   = [1.5]               # bar (was [1.3, 1.4, 1.5])
_VALIDATION_T   = [333.15]            # K   (was [323.15, 333.15, 343.15])
_VALIDATION_I   = [10, 30, 50]        # A   (was load_points)

T_SPAN = (0, 30)                      # was (0, 600); 30 s is enough for steady-state Ucell
MAX_STEP = 1e-1

t0_total = _time.perf_counter()
for RHC in _VALIDATION_RHC:
    for P_des in _VALIDATION_P:
        for T_des in _VALIDATION_T:
            cond_key = f"RHC{RHC}_P{P_des}_T{T_des}"
            print(f"\n=== {cond_key} ===")
            states_test = []           # list of dicts (one per successful current)
            i_kept      = []
            for I_LOAD in _VALIDATION_I:
                op = dict(operating_inputs)
                op["Phi_c_des"] = RHC
                op["Pa_des"]    = P_des * 1e5
                op["Pc_des"]    = P_des * 1e5
                op["Tfc"]       = T_des
                op["current_density"] = (lambda x, _I=I_LOAD: _I / parameters["Aact"])

                t0 = _time.perf_counter()
                try:
                    x_init = init_x_for("dynamic", op, parameters)
                    if RHC == 0:
                        x_init[model_dummy_var_names.index("Wc_inj") if False else 0] = 0.0
                    # Use the same initial state both for model construction and the solver
                    model = PEMFC_dyn(parameters, op, x_init)
                    # Override Wc_inj setpoint by name (works for non-zero RHC)
                    if RHC > 0:
                        idx_wc = model.solver_variable_names.index("Wc_inj")
                        x_init[idx_wc] = 4.0e-5
                    sol = solve_ivp(model.dxdt, T_SPAN, x_init,
                                    method="BDF", max_step=MAX_STEP)
                    if not sol.success:
                        print(f"  I = {I_LOAD:5.1f} A  ->  solver failed: {sol.message}")
                        continue
                    model._recovery(sol)
                    # Gather last-step values from both variables and ec_kinetics
                    states = {}
                    for var_name in model.variables:
                        v = model.variables[var_name]
                        states[var_name] = v[-1] if hasattr(v, "__len__") and len(v) else float("nan")
                    for var_name in model.ec_kinetics:
                        v = model.ec_kinetics[var_name]
                        states[var_name] = v[-1] if hasattr(v, "__len__") and len(v) else float("nan")
                    # Reject if Ucell is non-finite or unphysical
                    u = states.get("Ucell", float("nan"))
                    try:
                        u = float(u)
                    except Exception:
                        u = float("nan")
                    if not (np.isfinite(u) and 0.0 < u < 1.3):
                        print(f"  I = {I_LOAD:5.1f} A  ->  unphysical Ucell = {u}")
                        continue
                    states_test.append(states)
                    i_kept.append(I_LOAD)
                    print(f"  I = {I_LOAD:5.1f} A  ->  Ucell = {u:.3f} V"
                          f"   ({_time.perf_counter() - t0:.1f} s wall)")
                except Exception as exc:
                    print(f"  I = {I_LOAD:5.1f} A  ->  {type(exc).__name__}: {exc}")

            if not states_test:
                print(f"  -> {cond_key}: no successful points, skipping")
                continue

            # Pack into the same shape the downstream cells expect:
            # {cond_key: {"states": {var_name: [val_per_i]}}}
            states_profile = {var: [s.get(var, float("nan")) for s in states_test]
                              for var in states_test[0].keys()}
            states_profile["_i_kept"] = i_kept
            pola_tests_sim[cond_key] = {"states": states_profile, "i_kept": i_kept}

print(f"\nDone. {len(pola_tests_sim)} condition(s) with "
      f"successful points. Total wall time: {_time.perf_counter() - t0_total:.1f} s")



=== RHC0.5_P1.5_T333.15 ===


d:\MFC2024\model\model.py:972: RuntimeWarning: invalid value encountered in log
  Ueq = (E0 - 8.5e-4 * (x['Tccl'] - 298.15) + R * x['Tccl'] / (2 * F) * (np.log(R * x['Tccl'] * x['C_H2_acl'] / Pref) + 0.5 * np.log(R * x['Tccl'] * x['C_O2_ccl'] / Pref)))
c:\ProgramData\anaconda3\Lib\site-packages\scipy\integrate\_ivp\common.py:346: RuntimeWarning: overflow encountered in multiply
  h_new = (y[ind] + new_factor * y_scale[ind]) - y[ind]
c:\ProgramData\anaconda3\Lib\site-packages\scipy\integrate\_ivp\common.py:315: RuntimeWarning: overflow encountered in multiply
  h = (y + factor * y_scale) - y
c:\ProgramData\anaconda3\Lib\site-packages\scipy\integrate\_ivp\common.py:345: RuntimeWarning: overflow encountered in multiply
  new_factor = NUM_JAC_FACTOR_INCREASE * factor[ind]
c:\ProgramData\anaconda3\Lib\site-packages\scipy\integrate\_ivp\common.py:367: RuntimeWarning: overflow encountered in multiply
  factor[max_diff < NUM_JAC_DIFF_SMALL * scale] *= NUM_JAC_FACTOR_INCREASE
d:\MFC2024\model\m

  I =  10.0 A  ->  Ucell = 0.878 V   (54.5 s wall)
  I =  30.0 A  ->  unphysical Ucell = nan
